# HCP Brain Age Prediction with SpectralViT

Refactor of `ixi.ipynb` (IXI sex **classification**) into HCP **age regression**.

What changed, at a glance:
- **Data**: `load_ixi_data` (sex label, flat file layout) → `load_hcp_data` (continuous
  age in years, recursive search to handle HCP's nested per-subject directories and
  both bracketed and exact-age CSV formats). See `loader.py` for details.
- **Loss**: `BCEWithLogitsLoss` → `MSELoss`. No model architecture changes were
  needed — `SpectralViT`/`SpatialViT`/`AttentionUNet` already return a raw scalar
  per sample when `use_sigmoid=False` (their default), which is exactly a
  regression output.
- **Target scaling**: age is standardized (mean/std fit on the *train* fold only)
  before training, and predictions are converted back to years before any metric
  is computed. This keeps `OneCycleLR`/`AdamW` well-behaved regardless of whether
  your cohort spans 8–100 years or 22–37 years.
- **Metrics**: Accuracy/Sensitivity/Specificity/AUC → MAE, RMSE, R², Pearson r
  (the standard brain-age quartet).
- **Significance testing**: McNemar's test / DeLong's test (classification-only)
  → a paired t-test on absolute errors, and a paired bootstrap test on the MAE
  difference (see `util.py`).
- **Hyperparameter search**: `SpectralViT_cv` / `SpatialViT_cv` (select by AUC) →
  `SpectralViT_cv_regression` / `SpatialViT_cv_regression` (select by MAE).

**Before running:** update `HCP_DIR`, `CSV_PATH`, and `FILE_GLOB` in the config
cell below to match your actual HCP release and preprocessing pipeline — file
layout and the age column format (bracketed vs. exact) both vary by release.
See the `load_hcp_data` docstring in `loader.py` for the common cases.

## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import OneCycleLR
import gc
import matplotlib.pyplot as plt
from util import (seed_everything, count_parameters, plot_all_model_losses_regression,
                   get_metrics_array_regression, paired_ttest_abs_error, paired_bootstrap_mae_test)
from networks import SpectralViT, SpatialViT, AttentionUNet, SwinTransformer
from loader import load_hcp_data
from validate import SpectralViT_cv_regression, SpatialViT_cv_regression

# Configuration
HCP_DIR = os.path.expanduser('~/SpectralViT/data/HCP_extracted/')
CSV_PATH = os.path.expanduser('~/SpectralViT/data/HCP_extracted/HCP_demographics.csv')

# Recursive glob (relative to HCP_DIR) used to find each subject's T1w volume.
# Adjust to match your release/pipeline -- see load_hcp_data's docstring for options:
#   'T1w_restore_brain.nii.gz'          HCP-YA, MNINonLinear space
#   'T1w_acpc_dc_restore_brain.nii.gz'  HCP-YA, native/ACPC space
#   '*_desc-preproc_T1w.nii.gz'         HCP-Aging/Development, BIDS derivatives
FILE_GLOB = 'T1w_restore_brain.nii.gz'

VOL_SIZE = 96
EPOCHS = 500
LR = 1e-4
BATCH_SIZE = 8
N_FOLDS = 5
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
seed_everything(0)


## 2. Load Data

In [ ]:
# Load data
X, Y, subject_ids = load_hcp_data(HCP_DIR, CSV_PATH, vol_size=VOL_SIZE, file_glob=FILE_GLOB)
X_flat = X.reshape(len(X), -1)

print(f"Loaded {len(X)} samples")
print(f"Volume shape: {X.shape}")
print(f"Age range: {Y.min():.1f}-{Y.max():.1f} yrs (mean {Y.mean():.1f} \u00b1 {Y.std():.1f})")
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=0)


## 3. Hyperparameter Selection\n\nSame two-stage search as the IXI notebook (best spatial patch size, then best number of PCA components), just scored by lowest held-out **MAE** instead of highest AUC.

In [ ]:
# 1. Config
FAST_EPOCHS = 20
FAST_LR = 1e-3
X_flat = X.reshape(len(X), -1)

# 2. Spatial Selection
PATCH_SIZE = SpatialViT_cv_regression(
    X=X, Y=Y,
    patch_candidates=[8, 12, 16],
    device=device,
    vol_size=VOL_SIZE,
    epochs=FAST_EPOCHS,
    lr=FAST_LR,
    physical_bs=2,
    effective_bs=8
)

# 3. Spectral Selection
N_COMP = SpectralViT_cv_regression(
    X_flat=X_flat, Y=Y,
    pca_candidates=[16, 32, 64, 128],
    device=device,
    epochs=FAST_EPOCHS,
    lr=FAST_LR
)


## 4. Train Models\n\n### Spectral ViT

In [ ]:
# Spectral ViT
spectral_vit_history = {'spectral_vit_loss': []}
oof_preds_spec = []
oof_y_true = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X_flat), 1):
    print(f"Starting Spectral ViT Fold {fold}...")
    torch.cuda.empty_cache()

    # Fit PCA on training data only
    pca = PCA(n_components=N_COMP, whiten=True).fit(X_flat[train_idx])

    # Standardize age on the TRAIN fold only; predictions are converted back to
    # years before any metric is computed, so all reported numbers stay in years.
    y_mean, y_std = Y[train_idx].mean(), Y[train_idx].std() + 1e-8

    # Transform data
    tr_pca = torch.from_numpy(pca.transform(X_flat[train_idx])).float()
    tr_y = torch.from_numpy((Y[train_idx] - y_mean) / y_std).float()
    ts_pca = torch.from_numpy(pca.transform(X_flat[test_idx])).float().to(device)
    ts_y = Y[test_idx]  # raw years, for metrics

    loader = DataLoader(TensorDataset(tr_pca, tr_y), batch_size=BATCH_SIZE, shuffle=True)

    # use_sigmoid=False (the default) means the model already returns a raw scalar --
    # exactly the regression output we want, no network changes required.
    model = SpectralViT(
        n_inputs=N_COMP,
        embed_dim=16,
        use_mode_weights=True
    ).to(device)

    opt = optim.AdamW(model.parameters(), lr=LR)
    sched = OneCycleLR(opt, max_lr=LR, steps_per_epoch=len(loader), epochs=EPOCHS)
    crit = nn.MSELoss()

    # Training loop
    spectral_vit_fold_losses = []
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss_spectral_vit = 0
        for b_pca, b_y in loader:
            b_pca, b_y = b_pca.to(device), b_y.to(device)
            opt.zero_grad()

            loss = crit(model(b_pca), b_y)

            loss.backward()
            epoch_loss_spectral_vit += loss.item()
            opt.step()
            sched.step()

        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                preds = model(ts_pca).cpu().numpy() * y_std + y_mean
                mae = mean_absolute_error(ts_y, preds)
                print(f"Fold {fold} | Epoch {epoch+1:3d} | Spec MAE: {mae:.2f} yrs")

        spectral_vit_fold_losses.append(epoch_loss_spectral_vit / len(loader))
    spectral_vit_history['spectral_vit_loss'].append(spectral_vit_fold_losses)

    # Final evaluation
    model.eval()
    with torch.no_grad():
        preds = model(ts_pca).cpu().numpy() * y_std + y_mean
        oof_preds_spec.append(preds)
        oof_y_true.append(ts_y)

# Get parameter count
params_spec = count_parameters(model)
print(f"Spectral ViT parameters: {params_spec:,}")


### Spatial ViT (Heavy)

In [ ]:
# Spatial ViT Configuration
PHYSICAL_BATCH_SIZE = 2
EFFECTIVE_BATCH_SIZE = 8
ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // PHYSICAL_BATCH_SIZE

spatial_vit_history = {'spatial_vit_loss': []}
oof_preds_spat = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    print(f"Starting Spatial ViT Fold {fold}...")
    torch.cuda.empty_cache()

    y_mean, y_std = Y[train_idx].mean(), Y[train_idx].std() + 1e-8

    # Data preparation
    tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
    tr_y = torch.from_numpy((Y[train_idx] - y_mean) / y_std).float()
    ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
    ts_y = Y[test_idx]

    loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=PHYSICAL_BATCH_SIZE, shuffle=True)

    model = SpatialViT(
        vol_size=VOL_SIZE,
        patch_size=PATCH_SIZE,
        embed_dim=128
    ).to(device)

    opt = optim.AdamW(model.parameters(), lr=LR)
    sched = OneCycleLR(opt, max_lr=LR, steps_per_epoch=len(loader)//ACCUMULATION_STEPS, epochs=EPOCHS)
    crit = nn.MSELoss()

    spatial_vit_fold_losses = []
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss_spatial_vit = 0
        for i, (b_vol, b_y) in enumerate(loader):
            b_vol, b_y = b_vol.to(device), b_y.to(device)

            # Forward pass (returns a raw scalar per sample)
            logits = model(b_vol)
            loss = crit(logits, b_y) / ACCUMULATION_STEPS
            loss.backward()
            epoch_loss_spatial_vit += loss.item() * ACCUMULATION_STEPS

            if (i + 1) % ACCUMULATION_STEPS == 0:
                opt.step()
                sched.step()
                opt.zero_grad()

        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                test_preds = []
                # Processing in chunks to avoid OOM
                for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
                    test_preds.append(model(chunk))
                preds = torch.cat(test_preds).cpu().numpy() * y_std + y_mean
                mae = mean_absolute_error(ts_y, preds)
                print(f"Fold {fold} | Epoch {epoch+1:3d} | Spat MAE: {mae:.2f} yrs")

        spatial_vit_fold_losses.append(epoch_loss_spatial_vit / len(loader))

    spatial_vit_history['spatial_vit_loss'].append(spatial_vit_fold_losses)

    # Out-of-fold predictions
    model.eval()
    with torch.no_grad():
        test_preds = []
        for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
            test_preds.append(model(chunk))
        oof_preds_spat.append(torch.cat(test_preds).cpu().numpy() * y_std + y_mean)

# Parameter count and summary
params_spat = count_parameters(model)
print(f"Spatial ViT parameters: {params_spat:,}")


### Compact Spatial ViT

In [ ]:
# Compact Spatial ViT (uses the same unified SpatialViT class, smaller config)
PHYSICAL_BATCH_SIZE = 2
EFFECTIVE_BATCH_SIZE = 8
ACCUMULATION_STEPS = EFFECTIVE_BATCH_SIZE // PHYSICAL_BATCH_SIZE

compact_spatial_vit_history = {'compact_spatial_vit_loss': []}
oof_preds_spat_matched = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X), 1):
    print(f"Starting Compact Spatial ViT Fold {fold}...")
    torch.cuda.empty_cache()

    y_mean, y_std = Y[train_idx].mean(), Y[train_idx].std() + 1e-8

    # Data Preparation
    tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
    tr_y = torch.from_numpy((Y[train_idx] - y_mean) / y_std).float()
    ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
    ts_y = Y[test_idx]

    loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=PHYSICAL_BATCH_SIZE, shuffle=True)

    # use_cls_token=False uses mean pooling, matching the original "Matched" behavior.
    model = SpatialViT(
        vol_size=VOL_SIZE,
        patch_size=12,
        embed_dim=12,
        n_heads=1,
        n_layers=1,
        is_2d=False,
        use_cls_token=False
    ).to(device)

    opt = optim.AdamW(model.parameters(), lr=LR)
    sched = OneCycleLR(opt, max_lr=LR, steps_per_epoch=len(loader)//ACCUMULATION_STEPS, epochs=EPOCHS)
    crit = nn.MSELoss()

    compact_spatial_vit_fold_losses = []
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss_compact_spatial_vit = 0
        for i, (b_vol, b_y) in enumerate(loader):
            b_vol, b_y = b_vol.to(device), b_y.to(device)

            logits = model(b_vol)
            loss = crit(logits, b_y) / ACCUMULATION_STEPS
            loss.backward()

            epoch_loss_compact_spatial_vit += loss.item() * ACCUMULATION_STEPS

            if (i + 1) % ACCUMULATION_STEPS == 0:
                opt.step()
                sched.step()
                opt.zero_grad()

        if (epoch + 1) % 10 == 0:
            model.eval()
            with torch.no_grad():
                test_preds = []
                for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
                    test_preds.append(model(chunk))
                preds = torch.cat(test_preds).cpu().numpy() * y_std + y_mean
                mae = mean_absolute_error(ts_y, preds)
                print(f"Fold {fold} | Epoch {epoch+1:3d} | Matched MAE: {mae:.2f} yrs")

        compact_spatial_vit_fold_losses.append(epoch_loss_compact_spatial_vit / len(loader))

    compact_spatial_vit_history['compact_spatial_vit_loss'].append(compact_spatial_vit_fold_losses)

    # Final Inference
    model.eval()
    with torch.no_grad():
        test_preds = []
        for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
            test_preds.append(model(chunk))
        oof_preds_spat_matched.append(torch.cat(test_preds).cpu().numpy() * y_std + y_mean)

params_match = count_parameters(model)
print(f"\nMatched Spatial ViT parameters: {params_match:,}")


### Swin ViT

In [ ]:
# Swin Transformer
oof_preds_swin = []
# NOTE: the original IXI notebook re-created `kf` here with a hardcoded n_splits=5.
# That's silently fine only when N_FOLDS == 5; for any other N_FOLDS it evaluates
# Swin (and, transitively, U-Net below) on different folds than every other model,
# which breaks the final side-by-side comparison. We reuse the one `kf` from Setup
# instead, so all five models are always scored on identical folds.
swin_history = {'swin_loss': []}

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    y_mean, y_std = Y[train_idx].mean(), Y[train_idx].std() + 1e-8

    tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
    tr_y = torch.from_numpy((Y[train_idx] - y_mean) / y_std).float()
    ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
    ts_y = Y[test_idx]

    # Regression has no class-imbalance concept, so a plain shuffled loader
    # replaces the IXI notebook's BalancedDataset here.
    train_loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=BATCH_SIZE, shuffle=True)

    # img_size=96, patch_size=8, res=12, divisible by window_size=6
    m_swin = SwinTransformer(img_size=X.shape[-1], patch_size=8, window_size=6).to(device)
    optimizer = optim.AdamW(m_swin.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    print(f"Starting Swin Fold {fold+1}...")
    fold_swin_losses = []
    for epoch in range(1, EPOCHS + 1):
        m_swin.train()
        epoch_loss_swin = 0
        for b_vol, b_y in train_loader:
            b_vol, b_y = b_vol.to(device), b_y.to(device)
            optimizer.zero_grad()
            l_swin = criterion(m_swin(b_vol + torch.randn_like(b_vol)*0.01), b_y)
            l_swin.backward()
            optimizer.step()
            epoch_loss_swin += l_swin.item()

        # Save average epoch loss
        fold_swin_losses.append(epoch_loss_swin / len(train_loader))
    swin_history['swin_loss'].append(fold_swin_losses)

    with torch.no_grad():
        m_swin.eval()
        preds = m_swin(ts_vol).cpu().numpy() * y_std + y_mean
        oof_preds_swin.append(preds)

    del m_swin; gc.collect(); torch.cuda.empty_cache()

# Results
p_swin_final = np.concatenate(oof_preds_swin)

params_swin = sum(p.numel() for p in
                   SwinTransformer(img_size=X.shape[-1], patch_size=8, window_size=6).to(device).parameters()
                   if p.requires_grad)
print(f"Swin Parameters: {params_swin:,}")


### Attention U-Net

In [ ]:
# U-net
unet_history = {'unet_loss': []}
oof_preds_unet = []

for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
    print(f"Starting Attention U-Net Fold {fold+1}...")
    torch.cuda.empty_cache()

    y_mean, y_std = Y[train_idx].mean(), Y[train_idx].std() + 1e-8

    # Data
    tr_vol = torch.from_numpy(X[train_idx]).unsqueeze(1).float()
    tr_y = torch.from_numpy((Y[train_idx] - y_mean) / y_std).float()
    ts_vol = torch.from_numpy(X[test_idx]).unsqueeze(1).float().to(device)
    ts_y = Y[test_idx]

    train_loader = DataLoader(TensorDataset(tr_vol, tr_y), batch_size=BATCH_SIZE, shuffle=True)

    m_unet = AttentionUNet(in_channels=1, base_channels=7).to(device)
    optimizer = optim.AdamW(m_unet.parameters(), lr=1e-4)
    criterion = nn.MSELoss()

    fold_unet_losses = []
    for epoch in range(1, EPOCHS + 1):
        m_unet.train()
        epoch_loss_unet = 0
        for b_vol, b_y in train_loader:
            b_vol, b_y = b_vol.to(device), b_y.to(device)
            optimizer.zero_grad()
            loss = criterion(m_unet(b_vol), b_y)
            loss.backward()
            optimizer.step()
            epoch_loss_unet += loss.item()

        fold_unet_losses.append(epoch_loss_unet / len(train_loader))
    unet_history['unet_loss'].append(fold_unet_losses)

    # Fast Inference
    m_unet.eval()
    with torch.no_grad():
        # Using a loop for inference to prevent OOM on the test set
        test_preds = []
        for chunk in torch.split(ts_vol, PHYSICAL_BATCH_SIZE):
            test_preds.append(m_unet(chunk))
        oof_preds_unet.append(torch.cat(test_preds).cpu().numpy() * y_std + y_mean)

params_unet = sum(p.numel() for p in m_unet.parameters())
print(f"U-Net Parameters: {params_unet:,}")


## 5. Compare Models

In [ ]:
plot_all_model_losses_regression(spectral_vit_history, spatial_vit_history, compact_spatial_vit_history,
                                  unet_history, swin_history, EPOCHS)


In [ ]:
# Compute overall metrics
y_final = np.concatenate(oof_y_true)
p_spec_final = np.concatenate(oof_preds_spec)
p_spat_final = np.concatenate(oof_preds_spat)
p_match_final = np.concatenate(oof_preds_spat_matched)
p_unet_final = np.concatenate(oof_preds_unet)
# p_swin_final was already concatenated in the Swin cell above

spec_metrics = get_metrics_array_regression(y_final, p_spec_final)
spat_metrics = get_metrics_array_regression(y_final, p_spat_final)
match_metrics = get_metrics_array_regression(y_final, p_match_final)
swin_metrics = get_metrics_array_regression(y_final, p_swin_final)
unet_metrics = get_metrics_array_regression(y_final, p_unet_final)

# Compute per-fold variation
spec_fold_metrics = [get_metrics_array_regression(oof_y_true[f], oof_preds_spec[f]) for f in range(N_FOLDS)]
spat_fold_metrics = [get_metrics_array_regression(oof_y_true[f], oof_preds_spat[f]) for f in range(N_FOLDS)]
match_fold_metrics = [get_metrics_array_regression(oof_y_true[f], oof_preds_spat_matched[f]) for f in range(N_FOLDS)]
swin_fold_metrics = [get_metrics_array_regression(oof_y_true[f], oof_preds_swin[f]) for f in range(N_FOLDS)]
unet_fold_metrics = [get_metrics_array_regression(oof_y_true[f], oof_preds_unet[f]) for f in range(N_FOLDS)]

spec_std = np.std(spec_fold_metrics, axis=0)
spat_std = np.std(spat_fold_metrics, axis=0)
match_std = np.std(match_fold_metrics, axis=0)
swin_std = np.std(swin_fold_metrics, axis=0)
unet_std = np.std(unet_fold_metrics, axis=0)

# Print
col_width = 22
print("\n" + "="*135)
print(f"{'Metric':<15} | {'Spectral ViT':^{col_width}} | {'Spat-ViT (H)':^{col_width}} | {'Spat-ViT (M)':^{col_width}} | {'Swin ViT':^{col_width}} | {'Attn U-Net':^{col_width}}")
print("-" * 135)

metric_names = ['MAE (yrs)', 'RMSE (yrs)', 'R^2', 'Pearson r']
for i, name in enumerate(metric_names):
    spec_str  = f"{spec_metrics[i]:.3f} \u00b1 {spec_std[i]:.3f}"
    spat_str  = f"{spat_metrics[i]:.3f} \u00b1 {spat_std[i]:.3f}"
    match_str = f"{match_metrics[i]:.3f} \u00b1 {match_std[i]:.3f}"
    swin_str  = f"{swin_metrics[i]:.3f} \u00b1 {swin_std[i]:.3f}"
    unet_str  = f"{unet_metrics[i]:.3f} \u00b1 {unet_std[i]:.3f}"
    print(f"{name:<15} | {spec_str:^{col_width}} | {spat_str:^{col_width}} | {match_str:^{col_width}} | {swin_str:^{col_width}} | {unet_str:^{col_width}}")

print("-" * 135)
print(f"{'Parameters':<15} | {params_spec:^{col_width},} | {params_spat:^{col_width},} | {params_match:^{col_width},} | {params_swin:^{col_width},} | {params_unet:^{col_width},}")
print("="*135)


### Predicted vs. chronological age

Added relative to the IXI notebook: since there's no ROC curve equivalent for
regression, a predicted-vs-actual scatter plot (with the identity line) is the
standard sanity check for a brain-age model -- systematic bowing away from the
diagonal usually signals age-regression-to-the-mean rather than a genuine
biological effect.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4.2), sharex=True, sharey=True)
model_preds = [
    ("Spectral ViT", p_spec_final),
    ("Spat-ViT (H)", p_spat_final),
    ("Spat-ViT (M)", p_match_final),
    ("Swin ViT", p_swin_final),
    ("Attn U-Net", p_unet_final),
]
lo, hi = y_final.min(), y_final.max()

for ax, (name, preds) in zip(axes, model_preds):
    ax.scatter(y_final, preds, alpha=0.5, s=18)
    ax.plot([lo, hi], [lo, hi], 'k--', lw=1)
    ax.set_title(name, fontsize=12)
    ax.set_xlabel("Chronological age (yrs)")
axes[0].set_ylabel("Predicted age (yrs)")

plt.suptitle("Out-of-fold predicted vs. chronological age", fontsize=14)
plt.tight_layout()
plt.show()


## 6. Save Results

In [ ]:
# %% [markdown]
# ### Master Save Cell
# Saves everything: RNG states, OOF predictions, and metadata.

import torch
import numpy as np
import random

# Collect all current results
master_state = {
    # 1. THE DATA RESULTS (The most important for saving time)
    'results': {
        'oof_preds_spec': oof_preds_spec,
        'oof_preds_spat': oof_preds_spat,
        'oof_preds_spat_matched': oof_preds_spat_matched,
        'oof_preds_swin': oof_preds_swin,
        'oof_preds_unet': oof_preds_unet,
        'oof_y_true': oof_y_true,
    },

    # 2. TRAINING HISTORIES (For plots)
    'histories': {
        'spec': spectral_vit_history,
        'spat': spatial_vit_history,
        'match': compact_spatial_vit_history,
        'swin': swin_history,
        'unet': unet_history
    },

    # 3. CONFIGURATION
    'config': {
        'task': 'age_regression',
        'N_COMP': N_COMP,
        'PATCH_SIZE': PATCH_SIZE,
        'N_FOLDS': N_FOLDS,
        'LR': LR,
        'VOL_SIZE': VOL_SIZE,
        'EPOCHS': EPOCHS,
        'FILE_GLOB': FILE_GLOB
    },

    # 4. RANDOM STATES (To guarantee exact retraining results)
    'rng_state': {
        'python': random.getstate(),
        'numpy': np.random.get_state(),
        'torch_cpu': torch.get_rng_state(),
        'torch_gpu': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }
}

torch.save(master_state, 'spectral_vit_hcp_age_master_state.pth')
print("\u2705 MASTER STATE SAVED: Predictions, Histories, and RNG states are secured.")


## 7. Statistical Significance

McNemar's test and DeLong's test are classification-specific (they operate on
predicted class labels and ROC curves), so they're replaced here with:
- **Paired t-test on absolute errors** -- tests whether one model's per-subject
  errors are systematically larger/smaller than another's on the same held-out
  subjects (the regression analog of McNemar's paired-error comparison).
- **Paired bootstrap test on the MAE difference** -- resamples the held-out
  subjects (not the errors independently) to get a p-value for the MAE gap
  between two models, respecting the fact that both models were scored on the
  same subjects (the regression analog of DeLong's test for correlated AUCs).

In [ ]:
# --- Execution of Significance Comparisons (baseline: Spectral ViT) ---
models_to_compare = {
    "Spat-ViT (H)": p_spat_final,
    "Spat-ViT (M)": p_match_final,
    "Swin ViT": p_swin_final,
    "Attn U-Net": p_unet_final,
}

print("\n" + "=" * 100)
print(f"{'Statistical Significance Testing (Baseline: Spectral ViT)':<40}")
print("-" * 100)
print(
    f"{'Comparison Model':<16} | {'Paired t-test (Abs. Error)':<32} |"
    f" {'Paired Bootstrap (MAE Diff.)':<34}"
)
print("-" * 100)

for model_name, p_other in models_to_compare.items():
    t_stat, t_p = paired_ttest_abs_error(y_final, p_spec_final, p_other)
    mae_diff, boot_p = paired_bootstrap_mae_test(y_final, p_spec_final, p_other)

    t_str = f"t = {t_stat:.2f} (p = {t_p:.4f})"
    b_str = f"\u0394MAE = {mae_diff:+.2f} yrs (p = {boot_p:.4f})"

    print(f"{model_name:<16} | {t_str:<32} | {b_str:<34}")

print("=" * 100)
